# 01 - Audit Data DNS (PeDaS 2026 Final)
Sumber: `data/raw/sample-dns-30min.csv`. Jalankan sel berurutan dari atas ke bawah.
Satu baris = satu pesan DNS (query **atau** respons), bukan satu pengguna.

## 1. Setup

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Jalankan notebook dari folder notebooks/, project root = satu tingkat di atas
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data/raw/sample-dns-30min.csv"
PQ = ROOT / "data/parquet/dns.parquet"
OUT = ROOT / "outputs"
PQ.parent.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("PRAGMA threads=4")   # naikkan sesuai jumlah core

def q(sql):
    """Jalankan SQL, kembalikan DataFrame."""
    return con.execute(sql).df()

print("File ada:", RAW.exists(), "| ukuran GB:", round(RAW.stat().st_size / 1e9, 2))

File ada: True | ukuran GB: 2.4


## 2. Konversi CSV ke Parquet (sekali saja)
Kalau muncul error parsing, ganti `ignore_errors=false` jadi `true`, lalu catat jumlah baris yang hilang sebagai bagian audit.

In [2]:
if not PQ.exists():
    con.execute(f"""
    COPY (
        SELECT * FROM read_csv('{RAW.as_posix()}',
                               header=true, sample_size=-1,
                               ignore_errors=false)
    ) TO '{PQ.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print("Parquet dibuat.")
else:
    print("Parquet sudah ada, lewati.")

print("Ukuran parquet (GB):", round(PQ.stat().st_size / 1e9, 2))
con.execute(f"CREATE OR REPLACE VIEW dns AS SELECT * FROM '{PQ.as_posix()}'")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parquet dibuat.
Ukuran parquet (GB): 0.26


## 3. Jumlah baris dan kolom

In [3]:
n_rows = q("SELECT COUNT(*) AS n FROM dns")["n"][0]
n_cols = len(q("DESCRIBE dns"))
print(f"Jumlah record: {n_rows:,} | Jumlah kolom: {n_cols}")

Jumlah record: 11,712,623 | Jumlah kolom: 34


## 4. Skema dan tipe data

In [4]:
schema = q("DESCRIBE dns")[["column_name", "column_type"]]
schema

,column_name,column_type
0,ts,DOUBLE
1,ts_iso,TIMESTAMP WITH TIME ZONE
2,ip_ver,BIGINT
3,proto,VARCHAR
4,src_ip,VARCHAR
5,src_port,BIGINT
6,dst_ip,VARCHAR
7,dst_port,BIGINT
8,frame_len,BIGINT
9,dns_len,BIGINT


## 5. Missing value per kolom (NULL dan string kosong)
Kolom seperti `qname`, `answers`, dan `ecs` bisa kosong secara wajar. Bedakan "tidak berlaku" dari "data hilang".

In [5]:
cols = schema["column_name"].tolist()
types = dict(zip(schema.column_name, schema.column_type))

parts = []
for c in cols:
    empty = f"SUM(CASE WHEN TRIM(CAST(\"{c}\" AS VARCHAR)) = '' THEN 1 ELSE 0 END)"
    parts.append(f"""SELECT '{c}' AS kolom,
                     SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END) AS n_null,
                     {empty} AS n_kosong,
                     COUNT(DISTINCT "{c}") AS n_unik
                     FROM dns""")

miss = q(" UNION ALL ".join(parts))
miss["pct_null"] = miss["n_null"] / n_rows * 100
miss["pct_kosong"] = miss["n_kosong"] / n_rows * 100
miss["pct_tidak_terisi"] = miss["pct_null"] + miss["pct_kosong"]
miss = miss.sort_values("pct_tidak_terisi", ascending=False)
miss

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,kolom,n_null,n_kosong,n_unik,pct_null,pct_kosong,pct_tidak_terisi
33,answers,"11,646,828.0000",0.0000,3881,99.4383,0.0000,99.4383
32,ecs_scope,"11,616,841.0000",0.0000,8,99.1822,0.0000,99.1822
31,ecs,"11,616,841.0000",0.0000,4733,99.1822,0.0000,99.1822
29,edns_udpsize,"423,003.0000",0.0000,68,3.6115,0.0000,3.6115
27,qclass,"4,606.0000",0.0000,2,0.0393,0.0000,0.0393
24,qname,"4,606.0000",0.0000,2243284,0.0393,0.0000,0.0393
25,qtype,"4,606.0000",0.0000,56,0.0393,0.0000,0.0393
26,qtype_name,"4,606.0000",0.0000,56,0.0393,0.0000,0.0393
1,ts_iso,0.0000,0.0000,11684319,0.0000,0.0000,0.0000
28,edns,0.0000,0.0000,2,0.0000,0.0000,0.0000


## 6. Cakupan waktu dan zona waktu
Lihat: (1) `ts` epoch detik atau format lain, (2) `ts_iso` berakhiran `Z`/`+00:00` (UTC) atau `+07:00` (WIB). Zona waktu wajib ditulis di laporan.

In [6]:
print(q("""
SELECT MIN(ts) AS ts_min, MAX(ts) AS ts_max,
       MIN(ts_iso) AS iso_min, MAX(ts_iso) AS iso_max
FROM dns
""").T)

print(q("SELECT ts, ts_iso FROM dns LIMIT 5"))

                                        0
ts_min                 1,787,125,797.6498
ts_max                 1,787,127,597.5664
iso_min  2026-08-19 14:49:57.649752+07:00
iso_max  2026-08-19 15:19:57.566419+07:00
                  ts                           ts_iso
0 1,787,125,797.6498 2026-08-19 14:49:57.649752+07:00
1 1,787,125,797.6498 2026-08-19 14:49:57.649823+07:00
2 1,787,125,797.6500 2026-08-19 14:49:57.649992+07:00
3 1,787,125,797.6501 2026-08-19 14:49:57.650128+07:00
4 1,787,125,797.6515 2026-08-19 14:49:57.651486+07:00


## 7. Ekspresi waktu (sesuaikan dari hasil sel 6)
Durasi harusnya sekitar 1.800 detik (30 menit). Kalau jauh berbeda, cek lagi satuan `ts`.

In [7]:
# Kalau ts = epoch detik (float/int), ini benar:
TS_EXPR = "to_timestamp(CAST(ts AS DOUBLE))"

# Kalau ts berupa string waktu, pakai ini:
# TS_EXPR = "CAST(ts_iso AS TIMESTAMPTZ)"

print(q(f"""
SELECT MIN({TS_EXPR}) AS awal, MAX({TS_EXPR}) AS akhir,
       date_diff('second', MIN({TS_EXPR}), MAX({TS_EXPR})) AS durasi_detik
FROM dns
"""))

                              awal                            akhir  durasi_detik
0 2026-08-19 14:49:57.649752+07:00 2026-08-19 15:19:57.566419+07:00          1800


## 8. Duplikasi
Duplikat persis di data paket belum tentu error. Catat angkanya dan jelaskan keputusanmu (dibuang atau dipertahankan).

In [8]:
key_cols = ", ".join(f'"{c}"' for c in cols)
dup = q(f"""
SELECT COUNT(*) AS n_grup_duplikat, COALESCE(SUM(n - 1), 0) AS n_baris_kelebihan
FROM (SELECT COUNT(*) AS n FROM dns GROUP BY {key_cols} HAVING COUNT(*) > 1)
""")
print(dup)
print(f"Persen baris duplikat persis: {dup['n_baris_kelebihan'][0] / n_rows * 100:.4f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_grup_duplikat  n_baris_kelebihan
0                0             0.0000
Persen baris duplikat persis: 0.0000%


## 9. Komposisi query vs respons (`qr`)
`qr=0` query, `qr=1` respons. Volume permintaan = jumlah `qr=0`, bukan total baris.

In [9]:
qr_tab = q("""
SELECT qr, COUNT(*) AS n_pesan,
       COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct_dari_semua_pesan
FROM dns GROUP BY qr ORDER BY qr
""")
qr_tab

,qr,n_pesan,pct_dari_semua_pesan
0,0,5861701,50.0460
1,1,5850922,49.9540


## 10. Komposisi jaringan (`ip_ver`, `proto`, `opcode`)

In [10]:
for c in ["ip_ver", "proto", "opcode"]:
    print(f"\n== {c} ==")
    print(q(f"""
    SELECT "{c}" AS nilai, COUNT(*) AS n,
           COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct
    FROM dns GROUP BY 1 ORDER BY n DESC
    """))


== ip_ver ==
   nilai        n     pct
0      6  8482271 72.4199
1      4  3230352 27.5801

== proto ==
  nilai         n     pct
0   udp  11202232 95.6424
1   tcp    510391  4.3576

== opcode ==
   nilai         n     pct
0      0  11711379 99.9894
1      4       656  0.0056
2      5       588  0.0050


## 11. Distribusi `rcode` (hanya respons, `qr=1`)
Umumnya 0 = NOERROR, 2 = SERVFAIL, 3 = NXDOMAIN, 5 = REFUSED. Cek nilai yang benar-benar muncul di datamu (RFC 1035 bagian 4.1).

In [11]:
n_resp = q("SELECT COUNT(*) AS n FROM dns WHERE qr = 1")["n"][0]
print(f"Denominator: record respons (qr=1) = {n_resp:,}")

rcode_tab = q("""
SELECT rcode, COUNT(*) AS n,
       COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct_dari_respons
FROM dns WHERE qr = 1
GROUP BY rcode ORDER BY n DESC
""")
rcode_tab

Denominator: record respons (qr=1) = 5,850,922


,rcode,n,pct_dari_respons
0,0,5173188,88.4166
1,3,675229,11.5406
2,1,2155,0.0368
3,4,294,0.0050
4,5,37,0.0006
5,2,18,0.0003
6,9,1,0.0000


## 12. Tipe query (hanya `qr=0`)

In [12]:
print(q("""
SELECT qtype, qtype_name, COUNT(*) AS n,
       COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct_dari_query
FROM dns WHERE qr = 0
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20
"""))

    qtype qtype_name        n  pct_dari_query
0       1          A  2762209         47.1230
1       2         NS  1852410         31.6019
2      28       AAAA   505576          8.6251
3      43         DS   494324          8.4331
4      16        TXT    59446          1.0141
5      15         MX    58598          0.9997
6      65      HTTPS    57854          0.9870
7       5      CNAME    22774          0.3885
8      48     DNSKEY    15290          0.2608
9       6        SOA    13981          0.2385
10     33        SRV     7721          0.1317
11    257        CAA     5553          0.0947
12   <NA>       None     2148          0.0366
13    255        ANY     1446          0.0247
14     39      DNAME     1001          0.0171
15     12        PTR      516          0.0088
16    251       IXFR      329          0.0056
17     99        SPF      108          0.0018
18     35      NAPTR       76          0.0013
19     38     TYPE38       47          0.0008


## 13. Ukuran pesan

In [13]:
print(q("""
SELECT qr,
       COUNT(*) AS n,
       AVG(frame_len) AS frame_mean,
       quantile_cont(frame_len, [0.5, 0.95, 0.99]) AS frame_p50_p95_p99,
       MAX(frame_len) AS frame_max,
       AVG(dns_len) AS dns_mean,
       quantile_cont(dns_len, [0.5, 0.95, 0.99]) AS dns_p50_p95_p99,
       MAX(dns_len) AS dns_max
FROM dns GROUP BY qr ORDER BY qr
"""))

   qr        n  frame_mean        frame_p50_p95_p99  frame_max  dns_mean          dns_p50_p95_p99  dns_max
0   0  5861701    104.2438    [106.0, 124.0, 139.0]        381   46.6752       [44.0, 62.0, 72.0]      294
1   1  5850922    582.4761  [488.0, 1147.0, 1946.0]       2479  524.9315  [426.0, 1096.0, 1871.0]     2411


## 14. Kelengkapan EDNS
Kolom `do` diberi tanda kutip karena `DO` adalah kata kunci SQL.

In [14]:
print(q("""
SELECT qr,
       COUNT(*) AS n,
       SUM(CASE WHEN edns IS NOT NULL AND CAST(edns AS VARCHAR) <> '' THEN 1 ELSE 0 END) AS n_edns_terisi,
       SUM(CASE WHEN ecs  IS NOT NULL AND CAST(ecs  AS VARCHAR) <> '' THEN 1 ELSE 0 END) AS n_ecs_terisi
FROM dns GROUP BY qr ORDER BY qr
"""))

print(q("""
SELECT edns, edns_udpsize, "do", COUNT(*) AS n
FROM dns GROUP BY 1,2,3 ORDER BY n DESC LIMIT 15
"""))

   qr        n  n_edns_terisi  n_ecs_terisi
0   0  5861701 5,861,701.0000   95,782.0000
1   1  5850922 5,850,922.0000        0.0000
    edns  edns_udpsize  do        n
0      1          1232   1  6071503
1      1          1400   1  2781810
2      1          1452   1   699718
3      1          1232   0   589719
4      0          <NA>   0   423003
5      1          1224   1   253203
6      1          4096   0   135456
7      1          4096   1   133092
8      1          1440   0   118850
9      1          1400   0   113244
10     1          1440   1   102961
11     1          1220   1    89774
12     1          1220   0    63174
13     1          1410   1    38218
14     1          4000   1    26132


## 15. Volume per menit (query, respons, total)

In [15]:
vol = q(f"""
SELECT date_trunc('minute', {TS_EXPR}) AS menit,
       COUNT(*) AS total_pesan,
       SUM(CASE WHEN qr = 0 THEN 1 ELSE 0 END) AS n_query,
       SUM(CASE WHEN qr = 1 THEN 1 ELSE 0 END) AS n_respons
FROM dns GROUP BY 1 ORDER BY 1
""")
vol

,menit,total_pesan,n_query,n_respons
0,2026-08-19 14:49:00+07:00,14740,"7,373.0000","7,367.0000"
1,2026-08-19 14:50:00+07:00,393819,"196,980.0000","196,839.0000"
2,2026-08-19 14:51:00+07:00,343116,"171,622.0000","171,494.0000"
3,2026-08-19 14:52:00+07:00,329240,"164,676.0000","164,564.0000"
4,2026-08-19 14:53:00+07:00,326427,"163,291.0000","163,136.0000"
5,2026-08-19 14:54:00+07:00,336932,"168,895.0000","168,037.0000"
6,2026-08-19 14:55:00+07:00,363142,"182,226.0000","180,916.0000"
7,2026-08-19 14:56:00+07:00,339629,"169,865.0000","169,764.0000"
8,2026-08-19 14:57:00+07:00,436071,"218,273.0000","217,798.0000"
9,2026-08-19 14:58:00+07:00,384998,"192,993.0000","192,005.0000"


## 16. Cek konsistensi cepat
Jika query tidak selalu ke port 53 atau respons tidak selalu dari port 53, catat: ini memengaruhi pencocokan query-respons nanti.

In [16]:
print("Query dengan rcode bukan 0 (tidak lazim):")
print(q("SELECT COUNT(*) AS n FROM dns WHERE qr = 0 AND rcode <> 0"))

print("\nPort 53 sebagai src/dst per qr:")
print(q("""
SELECT qr,
       SUM(CASE WHEN dst_port = 53 THEN 1 ELSE 0 END) AS dst53,
       SUM(CASE WHEN src_port = 53 THEN 1 ELSE 0 END) AS src53,
       COUNT(*) AS n
FROM dns GROUP BY qr ORDER BY qr
"""))

Query dengan rcode bukan 0 (tidak lazim):
      n
0  2146

Port 53 sebagai src/dst per qr:
   qr          dst53          src53        n
0   0 5,861,701.0000        31.0000  5861701
1   1        48.0000 5,850,903.0000  5850922


## 17. Simpan ringkasan audit
Berisi agregat saja (tanpa IP atau `qname` mentah), aman untuk didiskusikan dengan AI.

In [17]:
summary = {"n_rows": int(n_rows), "n_cols": int(n_cols)}
(OUT / "audit_ringkasan").mkdir(exist_ok=True)
schema.to_csv(OUT / "audit_ringkasan/skema.csv", index=False)
miss.to_csv(OUT / "audit_ringkasan/missing.csv", index=False)
qr_tab.to_csv(OUT / "audit_ringkasan/qr.csv", index=False)
rcode_tab.to_csv(OUT / "audit_ringkasan/rcode.csv", index=False)
vol.to_csv(OUT / "audit_ringkasan/volume_per_menit.csv", index=False)
pd.Series(summary).to_csv(OUT / "audit_ringkasan/umum.csv")
print("Tersimpan di outputs/audit_ringkasan/")

Tersimpan di outputs/audit_ringkasan/
